In [2]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd

from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torchvision import transforms, datasets
import torch.optim as optim
from PIL import Image
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import os
import requests

In [3]:
df = pd.read_csv('train.csv')
df.head()

C:\Users\Eddie\AppData\Local\Temp\ipykernel_51772\189615863.py:1: DtypeWarning: Columns (0: sub-region, 1: unique_sub-region) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train.csv')


,id,latitude,longitude,thumb_original_url,country,sequence,captured_at,lon_bin,lat_bin,cell,...,quadtree_10_50000,quadtree_10_12500,quadtree_10_500,quadtree_10_2500,unique_region,unique_sub-region,unique_city,unique_country,creator_username,creator_id
0,3859149887465501,-43.804769,-176.614093,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,1qOCfogkQ5uE4ZvEf_4n2A,1547559856000,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
1,574181207305439,-43.796611,-176.660483,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,5rdtj6tui6qdlg8q812ul2,1567257056000,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
2,333574322129026,-43.818092,-176.578383,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,nJcst1M2bFUxSOA54CZiy9,1531231715132,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
3,636305258168031,-44.052910,-176.633065,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,BGSwcf0pLCE5bMUWdKTeqk,1662645171414,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14
4,166741299029322,-43.748077,-176.329626,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,NZ,05pt8HKnLCf47UTGPrxuzB,1531143464444,0,8,"(0, 8)",...,0,0,0,0,Chatham Islands_NZ,NaN,Waitangi_NaN_Chatham Islands_NZ,NZ,roadroid,1.113362e+14


In [4]:
mask = df['unique_country'] == 'US'
us_set = df[mask]

states = us_set['unique_region'].unique()
print(states)

<ArrowStringArray>
[          'Hawaii_US',            'Texas_US',       'California_US',
          'Arizona_US',           'Nevada_US',             'Utah_US',
       'New Mexico_US',        'Louisiana_US',      'Mississippi_US',
          'Alabama_US',         'Colorado_US',         'Oklahoma_US',
         'Arkansas_US',           'Kansas_US',         'Missouri_US',
        'Tennessee_US',         'Kentucky_US',          'Wyoming_US',
           'Oregon_US',            'Idaho_US',          'Montana_US',
         'Nebraska_US',         'Illinois_US',          'Indiana_US',
             'Iowa_US',     'South Dakota_US',        'Wisconsin_US',
        'Minnesota_US',          'Florida_US',          'Georgia_US',
   'South Carolina_US',   'North Carolina_US',         'Virginia_US',
    'West Virginia_US',             'Ohio_US',         'Maryland_US',
 'Washington, D.C._US',     'Pennsylvania_US',         'Delaware_US',
       'New Jersey_US',         'New York_US',      'Connecticut_US',
 

In [5]:
print(df.columns)

Index(['id', 'latitude', 'longitude', 'thumb_original_url', 'country',
       'sequence', 'captured_at', 'lon_bin', 'lat_bin', 'cell', 'region',
       'sub-region', 'city', 'land_cover', 'road_index', 'drive_side',
       'climate', 'soil', 'dist_sea', 'quadtree_10_5000', 'quadtree_10_25000',
       'quadtree_10_1000', 'quadtree_10_50000', 'quadtree_10_12500',
       'quadtree_10_500', 'quadtree_10_2500', 'unique_region',
       'unique_sub-region', 'unique_city', 'unique_country',
       'creator_username', 'creator_id'],
      dtype='str')


In [6]:
us_ids = us_set['id'].tolist()

In [7]:
print(len(us_ids))
print(us_set['quadtree_10_500'].unique())

1218664
[2898 2899 2900 ... 8627 8628 8629]


In [9]:
state_count = us_set['unique_region'].value_counts()
print(state_count[:10])

unique_region
California_US    140653
Florida_US        79687
Texas_US          71072
Georgia_US        55162
Ohio_US           54483
Michigan_US       49042
Arizona_US        46104
Washington_US     41675
Colorado_US       40157
Utah_US           39326
Name: count, dtype: int64


In [10]:
for geo in ["quadtree_10_500", "quadtree_10_5000", "quadtree_10_25000", "quadtree_10_50000"]:
    print("Quadtree type: ", geo)
    print("Counts: ", us_set[geo].nunique(), "cells")
    print(us_set[geo].describe())

Quadtree type:  quadtree_10_500
Counts:  4359 cells
count    1.218664e+06
mean     5.456261e+03
std      1.452846e+03
min      2.898000e+03
25%      4.153000e+03
50%      5.483000e+03
75%      6.588000e+03
max      8.629000e+03
Name: quadtree_10_500, dtype: float64
Quadtree type:  quadtree_10_5000
Counts:  646 cells
count    1.218664e+06
mean     9.041913e+02
std      2.097024e+02
min      5.160000e+02
25%      7.220000e+02
50%      9.210000e+02
75%      1.061000e+03
max      1.356000e+03
Name: quadtree_10_5000, dtype: float64
Quadtree type:  quadtree_10_25000
Counts:  117 cells
count    1.218664e+06
mean     1.475108e+02
std      3.772514e+01
min      8.200000e+01
25%      1.170000e+02
50%      1.460000e+02
75%      1.780000e+02
max      2.340000e+02
Name: quadtree_10_25000, dtype: float64
Quadtree type:  quadtree_10_50000
Counts:  61 cells
count    1.218664e+06
mean     8.470693e+01
std      1.964371e+01
min      4.800000e+01
25%      6.900000e+01
50%      8.700000e+01
75%      1.000

In [11]:
us_set['quadtree_10_5000']

756112      516
756113      516
756114      516
756115      516
756116      516
           ... 
2333365    1356
2333366    1356
2333367    1356
2333368    1356
2333369    1356
Name: quadtree_10_5000, Length: 1218664, dtype: int64

In [71]:
# using quadtree 10 5000
# take 150 or 400 from each cell 
# 93k or 240k images in total, 4.5GB or 20GB

count = 400
cells = us_set['quadtree_10_5000'].unique()

us_ids_balanced = {} # create a dictionary where key is cell number and the value is a list of ids

for c in cells:
    mask = us_set['quadtree_10_5000'] == c
    cell_df = us_set[mask]
    if len(cell_df) >= count:   # if more values than 150 then take a random 150 sample
        cell_ids = cell_df.sample(count)['id'].tolist()
        
    else:                      # else take all
        cell_ids = cell_df.sample(len(cell_df))['id'].tolist()
            
    us_ids_balanced[c] = cell_ids

In [87]:
quads = us_ids_balanced.keys()
quad_set = set(quads)

us_ids_set = set()

for id_list in us_ids_balanced.values():
    for i in id_list:
        us_ids_set.add(i)

#for quad in quads:
 #   os.makedirs('./' + quad, exist_ok=True)

print(len(us_ids_set))
print(type(us_ids_balanced[600][0])) # check if the ids are ints or strs
print(list(us_ids_set)[:100])

240935
<class 'int'>
[198345305882625, 356369422614530, 190072142954498, 134280362065924, 1143527466074118, 1116888962170887, 513246163369991, 223056539156489, 239962411237388, 206534457950220, 1277977069355020, 321490672680974, 301373361487888, 468167024443411, 2981090272215064, 779219319652378, 138373248319515, 1767229873455132, 1213589882404898, 344955740487715, 378225896718372, 283530483466277, 119338060218415, 497493858058289, 766596031184946, 931968597360693, 269389301547064, 844806833045560, 1151932728606778, 812450259664958, 2782533121998911, 465007844786243, 2832371193741390, 2016872931786831, 851824442081358, 162179022585937, 1129343217565779, 323839929090131, 142911021252693, 197696118849621, 172281664897118, 227062989193313, 152334670233700, 156403636371557, 3117128571945063, 382577036230760, 159385029443692, 377484710248568, 1145502472601723, 187119020540027, 959744114688130, 186656853328009, 697112177672331, 296760038654094, 818388375437455, 481794253127824, 3381079677339

In [79]:
from pathlib import Path
import shutil
import zipfile

train_path = './datasets/osv5m/images/train'
train_dir = Path(train_path)
train_file_list = os.listdir(train_path)

test_path = './datasets/osv5m/images/test'
test_file_list = os.listdir(test_path)
test_dir = Path(test_path)

os.makedirs('us_balanced_images_more', exist_ok=True)

balanced_images_path = Path('us_balanced_images_more')

# save only the files needed for the final data set, remove the rest
# we will group the train and test for now and seperate later using a seperate splitting mechanism as to control split percentage

kept = 0
failed = 0
to_remove = 0

for base_dir in [train_dir, test_dir]:
    print('Using directory: ', base_dir)

    zip_files = list(base_dir.rglob('*.zip'))
    for zip_path in zip_files:
        #print('Using zip path: ', zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            #print('opened the zip file')
            for file in zip_ref.namelist():
                #print('got a list of files in the zip')
                if file.lower().endswith('.jpg'):
                    file_path = Path(file)
                    id_name = file_path.stem

                    try:
                        img_id = int(id_name)
                    except ValueError:
                        failed += 1
                        continue

                    if img_id in us_ids_set:
                        kept += 1
                        if kept % 1000 == 0:
                            print(f'KEPT {kept} IMAGES SO FAR')
                        destination = balanced_images_path
                        with zip_ref.open(file) as source:
                            target_path = destination / file_path.name
                            with open(target_path, 'wb') as target_file:
                                target_file.write(source.read())
                    else:
                        to_remove += 1

Using directory:  datasets\osv5m\images\train
KEPT 1000 IMAGES SO FAR
KEPT 2000 IMAGES SO FAR
KEPT 3000 IMAGES SO FAR
KEPT 4000 IMAGES SO FAR
KEPT 5000 IMAGES SO FAR
KEPT 6000 IMAGES SO FAR
KEPT 7000 IMAGES SO FAR
KEPT 8000 IMAGES SO FAR
KEPT 9000 IMAGES SO FAR
KEPT 10000 IMAGES SO FAR
KEPT 11000 IMAGES SO FAR
KEPT 12000 IMAGES SO FAR
KEPT 13000 IMAGES SO FAR
KEPT 14000 IMAGES SO FAR
KEPT 15000 IMAGES SO FAR
KEPT 16000 IMAGES SO FAR
KEPT 17000 IMAGES SO FAR
KEPT 18000 IMAGES SO FAR
KEPT 19000 IMAGES SO FAR
KEPT 20000 IMAGES SO FAR
KEPT 21000 IMAGES SO FAR
KEPT 22000 IMAGES SO FAR
KEPT 23000 IMAGES SO FAR
KEPT 24000 IMAGES SO FAR
KEPT 25000 IMAGES SO FAR
KEPT 26000 IMAGES SO FAR
KEPT 27000 IMAGES SO FAR
KEPT 28000 IMAGES SO FAR
KEPT 29000 IMAGES SO FAR
KEPT 30000 IMAGES SO FAR
KEPT 31000 IMAGES SO FAR
KEPT 32000 IMAGES SO FAR
KEPT 33000 IMAGES SO FAR
KEPT 34000 IMAGES SO FAR
KEPT 35000 IMAGES SO FAR
KEPT 36000 IMAGES SO FAR
KEPT 37000 IMAGES SO FAR
KEPT 38000 IMAGES SO FAR
KEPT 39000 IM

In [81]:
print(kept)

240935


In [97]:
moved = 0

for quad in quads:
    os.makedirs('./' + str(quad), exist_ok=True)

#there are two files, one with 93k images (150 cap per cell) and one with 240k images (400 cap per cell)
# us_balanced_images_more has 240k and is the one we will use
BASE = Path('us_balanced_images_more')

for img in os.listdir(BASE):
    img_path = Path(BASE / Path(img))
    found = False
    for quad in quads:
        try:
            image_id = int(img_path.stem)
        except ValueError:
            continue
        if image_id in us_ids_balanced[quad]:
            shutil.move(img_path, Path(str(quad)) / img_path.name)
            found = True
            moved += 1
            if moved % 1000 == 0:
                print('MOVED ', moved)
            break
    if not found:
        print('Not found', img)

MOVED  1000
MOVED  2000
MOVED  3000
MOVED  4000
MOVED  5000
MOVED  6000
MOVED  7000
MOVED  8000
MOVED  9000
MOVED  10000
MOVED  11000
MOVED  12000
MOVED  13000
MOVED  14000
MOVED  15000
MOVED  16000
MOVED  17000
MOVED  18000
MOVED  19000
MOVED  20000
MOVED  21000
MOVED  22000
MOVED  23000
MOVED  24000
MOVED  25000
MOVED  26000
MOVED  27000
MOVED  28000
MOVED  29000
MOVED  30000
MOVED  31000
MOVED  32000
MOVED  33000
MOVED  34000
MOVED  35000
MOVED  36000
MOVED  37000
MOVED  38000
MOVED  39000
MOVED  40000
MOVED  41000
MOVED  42000
MOVED  43000
MOVED  44000
MOVED  45000
MOVED  46000
MOVED  47000
MOVED  48000
MOVED  49000
MOVED  50000
MOVED  51000
MOVED  52000
MOVED  53000
MOVED  54000
MOVED  55000
MOVED  56000
MOVED  57000
MOVED  58000
MOVED  59000
MOVED  60000
MOVED  61000
MOVED  62000
MOVED  63000
MOVED  64000
MOVED  65000
MOVED  66000
MOVED  67000
MOVED  68000
MOVED  69000
MOVED  70000
MOVED  71000
MOVED  72000
MOVED  73000
MOVED  74000
MOVED  75000
MOVED  76000
MOVED  77000
MOVED  7

In [3]:
# Remove all empty quad files
import os

for name in os.listdir():
    if os.path.isdir(name) and len(os.listdir(name)) == 0:
        os.rmdir(name)